In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
import asyncio
import time

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        # model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_API_BASE)


In [2]:
#json格式的消息
messages = [
      {"role": "system", "content": "你是一个数学专家"},
      {"role": "user", "content": "100+100=?"},
      {"role": "assistant", "content": "200"},
      {"role": "user", "content": "我刚刚问了啥"},
]
response = model.invoke(messages)
print(response)

content='你刚刚问的是：**“100+100=?”**' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问题。用户问“我刚刚问了啥” 即“What did I just ask?”。需要回顾对话：用户上一条问“100+100=?” 我回答了200。现在用户问我刚刚问了啥。应回答：你刚刚问的是“100+100=?”。中文简洁。可以补充我回答了200。需要准确。 final.'} response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 47, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 78, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 47}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'df48930c-882d-4ff4-8d8d-2a2f5c8554c0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1b169-07fc-72d3-ba37-e99a340ffb8d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 92, 'total_tokens': 139, 'in

举例2

In [9]:
#json格式的消息
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
      SystemMessage(content="你是一个数学专家"),
      HumanMessage("100+100=?"),
      AIMessage("200"),
      HumanMessage("我刚刚问了啥")
]
response = model.invoke(messages)
print(response)

content='你刚刚问的是：**“100+100=?”**，我回答是 **200**。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问题。用户问“我刚刚问了啥”意思是“What did I just ask?” 需要回忆对话：用户之前问“100+100=?”，我回答了200。现在问刚刚问了啥。应简洁说明：你刚刚问的是“100+100=?”。可以加之前回答是200。注意用中文。最终。'} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 47, 'total_tokens': 143, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 74, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 47}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'a3fccc98-c55a-4b42-a4db-338448f3a11f', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1b16f-abff-774a-8253-6604de80fc7c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 96, 'total_tokens': 143

## HumanMessage 的使用

In [12]:
messages = [
      SystemMessage(
            "你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
      HumanMessage(
            content=" 我认为 1+1=2",
            name="Bob"
      ),
      HumanMessage(
            content=" 我认为 1+1>2",
            name="Tom"
      ),
      HumanMessage(
            content=" 请列出谁说了什么，不要判断对错。 ",
            name="audience")
]

response = model.invoke(messages)
print(response.content)

{"speakers":[{"name":"unknown","claim":"1+1=2"},{"name":"unknown","claim":"1+1>2"}]}


## AIMessage

In [ ]:
#json格式的消息
from rich import print as rprint

messages = [
      SystemMessage(content="你是一个数学专家"),
      HumanMessage("100+100=?"),

]
response = model.invoke(messages)
rprint(response)

## ToolMessage

In [28]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# 1. 直接用原生 OpenAI 客户端连接 DeepSeek
client = OpenAI(
      api_key=os.getenv("DEEPSEEK_API_KEY"),
      base_url=os.getenv("DEEPSEEK_API_BASE")
)

# 2. 构造消息（注意 reasoning_content 必须带上）
messages = [
      {"role": "user", "content": "北京天气如何"},
      {
            "role": "assistant",
            "content": "",
            "reasoning_content": "",  # 原生请求会严格保留它
            "tool_calls": [{
                  "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s",
                  "type": "function",  # 注意：原生API需要显式声明 type
                  "function": {
                        "name": "get_weather",
                        "arguments": '{"city": "北京"}'  # 原生API的 arguments 必须是 JSON 字符串
                  }
            }]
      },
      {
            "role": "tool",
            "content": "今天北京天气晴朗，万里无云~",
            "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
      }
]

# 3. 发起请求
response = client.chat.completions.create(
      model="deepseek-v4-flash",  # 直接用模型名，不用带前缀
      messages=messages
)

print(response.choices[0].message.content)

今天北京天气晴朗，万里无云～

不过这条天气信息比较简略，没有给出具体气温和风力。如果你需要更详细的数据（比如温度、空气质量、穿衣建议），可以告诉我，我再帮你查一下，或者你也可以参考气象局 App 的实时数据。


## 对话历史优化

In [30]:
def keep_recent_messages(messages, max_pairs=3):
      """
      保留最近的 N 轮对话

      max_pairs: 保留的对话轮数（每轮 = user + assistant ）
      """

      # 分离 system 和对话
      system_msgs = [m for m in messages if m.get("role") == "system"]
      conversation_msgs = [m for m in messages if m.get("role") != "system"]

      # 只保留最近的
      recent_msgs = conversation_msgs[-(max_pairs * 2):]

      # 返回： system + 最近对话
      return system_msgs + recent_msgs


long_conversation = [{"role": "system", "content": " 你是 Python 导师 "}
                     ]
# 第 1 轮
long_conversation.append({"role": "user", "content": " 什么是列表？用一句解释 "})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

# 第 2 轮
long_conversation.append({"role": "user", "content": " 列表和元组有什么区别？用一 句解释 "})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": " 什么是字典呢？用一句解释 "})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f" 原始消息数 : {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)

print(f" 优化后消息数 : {len(optimized)}")
print(f" 保留的内容 : system + 最近 2 轮对话 ")

# 添加新的用户问题
optimized.append({"role": "user", "content": " 我第一个问题问的是什么？ "})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复 : {response.content}")

 原始消息数 : 7
 优化后消息数 : 5
 保留的内容 : system + 最近 2 轮对话 

AI 回复 : 你第一个问题是：

> **列表和元组有什么区别？用一句解释**

我当时回答：列表用 `[]` 定义、可变；元组用 `()` 定义、不可变，后者更省内存且可作字典键。


## 多轮对话聊天机器人

In [32]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
import asyncio
import time

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        # model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_API_BASE)

# 3. 初始化消息列表
messages = [
      {
            "role": "system",
            "content": " 你是小谷姐姐，尚硅谷教育的数字员工，也是一名耐心、友好的智能助手。我会用自然、清晰的方式回答用户问题。 "}
]

# 4. 启动提示
print(f" ✨ 请输入问题，输入 {EXIT_WORD} 结束对话 \n")

# 5. 多轮对话主循环
# 轮次记录
i = 1
while True:
      print("\n", "=" * 10, f'-> 第 {i} 轮对话开始 <-', "=" * 10, "\n")
      user_input = input(" 🙋 请输入： ")

      # 退出判断
      if user_input.lower() == EXIT_WORD:
            print(" 🌙 对话已结束，欢迎下次再来！ ")
      break

# 追加用户消息
messages.append({"role": "user", "content": user_input})

# 流式输出模型回复
print(" 🧚 小谷姐姐： ", end="", flush=True)

reply_content = ""

# 优化历史记忆
memory_messages = keep_recent_messages(messages, max_pairs=MAX_PAIRS_HISTORY)
# 控制发送给模型的消息长度
for chunk in model.stream(memory_messages):
      if chunk.content:
            print(chunk.content, end="", flush=True)
reply_content += chunk.content

print("\n", "=" * 10, f'-> 第 {i} 轮对话结束 <-', "=" * 10, "\n")
i += 1

# 追加 AI 回复
messages.append({"role": "assistant", "content": reply_content})

 ✨ 请输入问题，输入 quit 结束对话 


 ========== -> 第 1 轮对话开始 <- ========== 

 🧚 小谷姐姐： 您说的 “pyhome” 我这边没对应上具体的课程或项目名称，方便您再确认一下吗？（是某个项目、还是 “Python” 的笔误？）

如果您其实是想问 **Python 学习路线**，可以先看这条通用主线（尚硅谷课程体系也是按这个思路走的）：

**① 语言基础**
变量/数据类型、运算符、流程控制、函数、模块与包、列表/字典/集合、字符串处理

**② 语言进阶**
面向对象（类、继承、多态）、异常处理、文件 IO、迭代器/生成器、装饰器、闭包、常用标准库

**③ 工程必备**
Linux 基础、Git、MySQL 数据库与 SQL、正则表达式、多线程/多进程/协程、网络编程（socket、HTTP）

**④ 选方向（挑 1~2 个深入）**
- **Web 后端**：Django / Flask / FastAPI + RESTful + Redis + 部署（Nginx、Docker）
- **数据分析**：NumPy → Pandas → Matplotlib → 可视化/报表
- **爬虫**：requests → 解析（XPath/BS4）→ Scrapy → 反爬与分布式
- **AI/机器学习**：sklearn → 深度学习框架（PyTorch）→ 项目实战

**⑤ 项目实战 + 工程化**
做 2~3 个完整项目（能讲清架构、数据库设计、部署），配合 Docker、CI、代码规范，这是简历上最值钱的部分。

建议节奏：基础 1~1.5 个月 → 进阶+工程 1.5 个月 → 方向深入 2 个月 → 项目 1~2 个月，边学边敲代码，别只看视频。

如果 “pyhome” 是您手上的具体课程/资料名，把全称或截图发我，我帮您对照着排一条更贴合的路线。
 ========== -> 第 1 轮对话结束 <- ========== 



content